# Tamper Signal quickstart: signed receipts for a pandas pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/welovejeff/tamper-evident-verification/blob/main/examples/quickstart.ipynb)

Tamper Signal signs a receipt at every stage of a data pipeline: a fingerprint of the data in, the code that ran, and the data out, plus control totals a human can read (row counts, sums, null counts). The receipts link into a chain, and one command checks it and answers with a traffic light: green (intact), yellow (verifies, but a human should look), or red (broken, at the exact link, with the numbers that moved). It can't tell you the data is right, but it can prove nobody changed it. This notebook runs the whole loop on a small pandas pipeline in about two minutes, with nothing to install locally. To see the status light in a browser instead, try the [live demo](https://tampersignal.com/demo.html); source and docs are on [GitHub](https://github.com/welovejeff/tamper-evident-verification).

## 1. Install

One package from PyPI. It gives you the `tamper-signal` command and the `tamper_signal` Python library.

In [1]:
%pip install -q tamper-signal

Note: you may need to restart the kernel to use updated packages.


## 2. A messy export

Twelve orders, the way exports usually arrive: one order has a blank `amount`, another is missing its `region`. We write it to `orders.csv`, which plays the part of the file someone downloaded from a sales tool.

In [2]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [f"A-{n}" for n in range(1001, 1013)],
    "region":   ["East", "West", "North", "South", "East", None,
                 "West", "North", "South", "East", "West", "North"],
    "amount":   [120.00, 85.50, 42.00, None, 310.25, 64.00,
                 19.99, 150.00, 73.40, 205.00, 99.95, 58.10],
})
orders.to_csv("orders.csv", index=False)
orders

,order_id,region,amount
0,A-1001,East,120.00
1,A-1002,West,85.50
2,A-1003,North,42.00
3,A-1004,South,NaN
4,A-1005,East,310.25
5,A-1006,NaN,64.00
6,A-1007,West,19.99
7,A-1008,North,150.00
8,A-1009,South,73.40
9,A-1010,East,205.00


## 3. Keys and the source receipt

`tamper-signal init` makes a signing keypair in `keys/`. `keys/signing.key` is the private key: keep it out of git (`init` adds `keys/` and `*.key` to `.gitignore` for you). `tamper-signal ingest` then signs the first receipt in the chain: the raw export, exactly as it arrived. `--origin` is free text for humans.

In [3]:
!tamper-signal init
!tamper-signal ingest orders.csv --origin "sample orders export" --key keys/signing.key --out receipts/

┌─ Tamper Signal ──────────────┐
│ signed receipts · green light │
└──────────────────────────────┘
  - keys: generated keys/signing.key and keys/signing.pub
  - .gitignore: added keys/, *.key
  - receipts: created receipts/

Next: tamper-signal ingest <export-file> --origin "..." --key keys/signing.key --out receipts/
Then wrap each transform with @receipt_step: https://tampersignal.com/docs/quickstart.html
Source: https://github.com/welovejeff/tamper-evident-verification


Ingested orders.csv
  evidence_hash 4a4d50c27df565d6d6686330a82f8937c1a302e13d1bfd58dc4bfafaf185e5a9
  semantic_hash bb150b51ec6a074fa72cf2c90e278f9b4860577c3682de4c702eacf347e58b27
  rows 12, columns 3
  source manifest -> receipts/000_source.json


## 4. Two wrapped pandas stages

`@receipt_step` wraps any function that takes and returns a DataFrame (or a list of dicts). Before running, it checks that the input descends from the last receipt in the chain and refuses if not. After running, it signs a receipt for the stage's output.

- `clean` drops rows with a blank amount or a missing region.
- `enrich` joins a small region-to-manager lookup. The lookup has a duplicated key (West appears twice), so the join quietly duplicates every West order. That is a bug on purpose: watch what the receipts do with it.

(`code_file` just labels where the code lives; in a notebook the default would be a temp file path.)

In [4]:
from tamper_signal import receipt_step

managers = pd.DataFrame({
    "region":  ["East", "West", "North", "South", "West"],
    "manager": ["Priya", "Marco", "Ana", "Sam", "Marco R."],  # West listed twice
})

@receipt_step(chain_dir="receipts/", key_path="keys/signing.key", code_file="quickstart.ipynb")
def clean(df):
    return df.dropna(subset=["amount", "region"])

@receipt_step(chain_dir="receipts/", key_path="keys/signing.key", code_file="quickstart.ipynb")
def enrich(df):
    return df.merge(managers, on="region", how="left")

raw = pd.read_csv("orders.csv")
cleaned = clean(raw)
dashboard = enrich(cleaned)
dashboard.to_csv("dashboard.csv", index=False)

print(f"orders.csv     {len(raw):>3} rows")
print(f"after clean    {len(cleaned):>3} rows")
print(f"dashboard.csv  {len(dashboard):>3} rows")

orders.csv      12 rows
after clean     10 rows
dashboard.csv   13 rows


## 5. Read the receipts

Receipts are plain JSON files in `receipts/`. Here are the control totals each stage signed. The row count goes 12, then 10, then 13: the enrich stage grew the data, and its own receipt says so.

In [5]:
import json
from pathlib import Path

for path in sorted(Path("receipts").glob("[0-9][0-9][0-9]_*.json")):
    receipt = json.loads(path.read_text())
    # the source receipt signs "control_totals"; each stage signs its "output_control_totals"
    totals = receipt.get("control_totals") or receipt["output_control_totals"]
    sums = ", ".join(f"{k}={v}" for k, v in totals["numeric_sums"].items())
    print(f"{path.name:<22} row_count={totals['row_count']:<3} {sums}")

000_source.json        row_count=12  amount=1228.19
001_clean.json         row_count=10  amount=1164.19
002_enrich.json        row_count=13  amount=1369.63


## 6. Verify the chain

`tamper-signal verify` walks every link and checks every signature. `--data` also checks the file the dashboard actually reads against the final receipt. The exit code is the light: 0 green, 1 red, 2 yellow.

In [6]:
!tamper-signal verify receipts/chain.json --pub keys/signing.pub --data dashboard.csv; echo "exit code: $?"

● GREEN
✓ CHAIN INTACT: 3 receipts, 2 transforms, final row_count 13


exit code: 0


The light is green, the data is clean. Green is about continuity: every stage's output fed the next stage, and nothing changed in between.

Totals moving across links is normal (a filter drops rows), so checking for it is off by default. For pipelines that should preserve totals, `--warn-drift` flags any movement as a caveat. Same chain, now yellow. The light is yellow, a human should look.

In [7]:
!tamper-signal verify receipts/chain.json --pub keys/signing.pub --data dashboard.csv --warn-drift; echo "exit code: $?"

● YELLOW
⚠ CHAIN VERIFIES, WITH CAVEATS: 3 receipts, 2 transforms, final row_count 13
  - totals drift at link 0 -> 1 (clean): row_count 12 -> 10 (-2), amount 1228.19 -> 1164.19 (-64), null_counts[amount] 1 -> 0 (-1), null_counts[region] 1 -> 0 (-1)
  - totals drift at link 1 -> 2 (enrich): row_count 10 -> 13 (+3), column_count 3 -> 4 (+1), amount 1164.19 -> 1369.63 (205.44)
  A human should look.


exit code: 2


## 7. Edit one number by hand

Now the thing receipts exist to catch: someone opens `dashboard.csv` and "fixes" one value after the pipeline ran. No stage signed that change, so the file no longer matches the final receipt. Expect red, with the exact total that moved.

In [8]:
edited = pd.read_csv("dashboard.csv")
print("before:", edited.loc[0, ["order_id", "amount"]].to_dict())
edited.loc[0, "amount"] = edited.loc[0, "amount"] + 100
edited.to_csv("dashboard.csv", index=False)
print("after: ", edited.loc[0, ["order_id", "amount"]].to_dict())

!tamper-signal verify receipts/chain.json --pub keys/signing.pub --data dashboard.csv; echo "exit code: $?"

before: {'order_id': 'A-1001', 'amount': 120.0}
after:  {'order_id': 'A-1001', 'amount': 220.0}


● RED
✗ DATA MISMATCH against final receipt (enrich)
  expected output hash 0e34...3d
  found    data hash   0735...14
  Control totals delta vs receipt: amount 1369.63 -> 1469.63 (100)
exit code: 1


## What this proves, and what it does not

- **It proves continuity.** Green means `dashboard.csv` descends from `orders.csv` through the signed stages, with nothing changed in between. Red names where the chain or the data stopped matching, and which totals moved.
- **It does not prove correctness.** A wrapped step is signed even when its logic is wrong. The `enrich` join above double counted every West order and still verified green. What the chain gives you is the evidence: that stage's own receipt shows `row_count` 10 to 13 and the amount sum jumping. Hashes say "broken," totals say "how broken," and a human still has to read them.

It can't tell you the data is right, but it can prove nobody changed it.

**Next steps**

- Quickstart docs: [tampersignal.com/docs/quickstart.html](https://tampersignal.com/docs/quickstart.html)
- Using a coding agent? Point it at [AGENTS.md](https://github.com/welovejeff/tamper-evident-verification/blob/main/AGENTS.md), the full integration runbook (ingest, wrap stages, verify, and put the status light on your dashboard).
- Watch the status light catch a tampered chain in your browser: [tampersignal.com/demo.html](https://tampersignal.com/demo.html)

If this is useful, a star on [GitHub](https://github.com/welovejeff/tamper-evident-verification) helps other people find it.